In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
import seaborn as sns
import gspread
from google.oauth2.service_account import Credentials

plt.rcParams.update({"figure.dpi":130,"font.family":"sans-serif",
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.3,"grid.linestyle":"--"})

COLORES_METODO = {
    "LinearRegression":"#2E86AB","Ridge":"#A23B72",
    "RandomForestRegressor":"#F18F01","GradientBoostingRegressor":"#4CAF50",
}
COLORES_ARCHIVO = {
    "df_diario.csv":"#264653","df_diario_1.csv":"#2A9D8F",
    "df_diario_2.csv":"#E9C46A","df_diario_3.csv":"#F4A261","df_diario_4.csv":"#E76F51",
}
LAGS_LABEL = {
    "df_diario.csv":"0 lags","df_diario_1.csv":"1 lag",
    "df_diario_2.csv":"2 lags","df_diario_3.csv":"3 lags","df_diario_4.csv":"4 lags",
}

In [ ]:
SCOPES = ["https://www.googleapis.com/auth/spreadsheets","https://www.googleapis.com/auth/drive"]
SERVICE_ACCOUNT_FILE = "../Modelos_semanales/credenciales_google.json"
SHEET_URL = "https://docs.google.com/spreadsheets/d/1Qc1Q1OIeSt1rMsGpo_xUQ90fhOG7RDeflyPn-cwVps0/edit?gid=1153965863#gid=1153965863"

def conectar_y_leer():
    creds  = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
    client = gspread.authorize(creds)
    ws     = client.open_by_url(SHEET_URL).get_worksheet(0)
    vals   = ws.get_all_values()
    return pd.DataFrame(vals[1:], columns=vals[0])

df_raw = conectar_y_leer()
print(f"Filas leidas: {len(df_raw)}")
print(df_raw.columns.tolist())

In [ ]:
def parse_num(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().replace("\xa0","").replace(" ","").replace("%","")
    if not s: return np.nan
    if "," in s and "." in s:
        s = s.replace(".","").replace(",",".") if s.rfind(",")>s.rfind(".") else s.replace(",","")
    elif "," in s: s = s.replace(",",".")
    return pd.to_numeric(s, errors="coerce")

In [ ]:
def limpiar(df):
    df = df[df["Estado"].astype(str).str.upper()=="DONE"].copy()
    for c in ["RMSE","RMSE_baseline","RMSE_zeros","MAE","MAE_baseline","MAE_zeros"]:
        df[c] = df[c].apply(parse_num)
    df["ETF"]  = df["Target"].astype(str).str.replace("target_","",regex=False)
    df["Lags"] = df["Archivo"].map(LAGS_LABEL).fillna("?")
    df["Mejora_vs_baseline"] = ((df["RMSE_baseline"]-df["RMSE"])/df["RMSE_baseline"]*100).round(2)
    df["Mejora_vs_zeros"]    = ((df["RMSE_zeros"]   -df["RMSE"])/df["RMSE_zeros"]   *100).round(2)
    df["Bate_baseline"] = df["RMSE"] < df["RMSE_baseline"]
    df["Bate_zeros"]    = df["RMSE"] < df["RMSE_zeros"]
    return df

In [ ]:
df = limpiar(df_raw)
print(f"Experimentos DONE: {len(df)}")

## RMSE medio por modelo y número de lags

In [ ]:
print("\n" + "="*70)
print("TABLA 1 — RMSE medio por Método y Archivo — Modelos diarios")
print("="*70)
tabla1 = df.groupby(["Archivo","Método"])[["RMSE","RMSE_baseline","RMSE_zeros"]].mean().round(5)
tabla1["Mejora_vs_baseline(%)"] = ((tabla1["RMSE_baseline"]-tabla1["RMSE"])/tabla1["RMSE_baseline"]*100).round(2)
tabla1["Mejora_vs_zeros(%)"]    = ((tabla1["RMSE_zeros"]   -tabla1["RMSE"])/tabla1["RMSE_zeros"]   *100).round(2)
print(tabla1.to_string())

Los tres modelos que funcionan bien (Ridge, Random Forest y Gradient Boosting) obtienen un RMSE muy parecido entre sí en todas las configuraciones de lags, con una mejora de alrededor del 31 % sobre el baseline. Esto quiere decir que los tres reducen el error de predicción de forma consistente sin importar cuántos lags se añadan.

La Regresión Lineal falla claramente: sin regularización, añadir más variables la hace empeorar hasta triplicar el error del baseline. No es una opción válida para este problema.

El dato más llamativo es que añadir lags apenas cambia los resultados de los modelos buenos. El RMSE con 0 lags y con 4 lags es casi el mismo, lo que indica que los retornos diarios pasados no aportan mucha información extra más allá de lo que ya recoge el período más reciente.

El benchmark de predicción cero tiene un RMSE muy similar al de los mejores modelos. Esto es normal: los retornos diarios son casi aleatorios, por lo que predecir cero es ya una estrategia razonablemente buena. El valor real del modelo está en mejorar sobre el baseline inercial, que es quien comete más error al copiar el último retorno observado.

## MAE medio por modelo y número de lags

In [ ]:
# TABLA 2 — MAE medio por Método y Archivo
print("\n" + "="*70)
print("TABLA 2 — MAE medio por Método y Archivo — Modelos diarios")
print("="*70)
tabla2 = df.groupby(["Archivo","Método"])[["MAE","MAE_baseline","MAE_zeros"]].mean().round(5)
tabla2["Mejora_vs_baseline(%)"] = ((tabla2["MAE_baseline"]-tabla2["MAE"])/tabla2["MAE_baseline"]*100).round(2)
tabla2["Mejora_vs_zeros(%)"]    = ((tabla2["MAE_zeros"]   -tabla2["MAE"])/tabla2["MAE_zeros"]   *100).round(2)
print(tabla2.to_string())

El MAE confirma lo mismo que el RMSE: Ridge, Random Forest y Gradient Boosting mejoran alrededor de un 28–30 % sobre el baseline en todas las configuraciones, y la Regresión Lineal vuelve a ser la peor opción.

Que tanto el RMSE como el MAE mejoren en la misma proporción es una buena señal: significa que el modelo no tiene errores muy grandes y puntuales que inflen el RMSE. Los errores son repartidos y moderados, algo esperable en predicción de retornos diarios.

El MAE de los mejores modelos equivale a un error promedio de unos 0.38–0.40 % de retorno, lo cual es razonable teniendo en cuenta que la volatilidad típica de estos ETFs ronda el 0.5–1.5 % en ese período.

## RMSE por modelo y configuración de lags — gráfico de barras

In [ ]:
import math
archivos_ord = ["df_diario.csv","df_diario_1.csv","df_diario_2.csv","df_diario_3.csv","df_diario_4.csv"]
arch_pres = [a for a in archivos_ord if a in df["Archivo"].unique()]
ncols=2; nrows=math.ceil(len(arch_pres)/ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(14,5.5*nrows), sharey=True)
axes = axes.flatten()
for ax, archivo in zip(axes, arch_pres):
    sub=df[df["Archivo"]==archivo]
    pm=sub.groupby("Método")["RMSE"].mean().reindex(list(COLORES_METODO.keys())).dropna()
    bl=sub["RMSE_baseline"].mean(); ze=sub["RMSE_zeros"].mean()
    x=range(len(pm)); cols=[COLORES_METODO.get(m,"#888") for m in pm.index]
    bars=ax.bar(x,pm.values,color=cols,edgecolor="white",width=0.6)
    ax.bar_label(bars,fmt="%.5f",fontsize=9,padding=3)
    ax.axhline(bl,color="#222",linestyle="--",linewidth=1.6)
    ax.axhline(ze,color="#7B2FBE",linestyle=":",linewidth=1.6)
    ax.annotate(f"Baseline\n{bl:.5f}",xy=(1.01,bl),xycoords=("axes fraction","data"),
        fontsize=9,color="#222",va="center",bbox=dict(boxstyle="round,pad=0.2",fc="white",ec="#222",alpha=0.85))
    ax.annotate(f"Zeros\n{ze:.5f}",xy=(1.01,ze),xycoords=("axes fraction","data"),
        fontsize=9,color="#7B2FBE",va="center",bbox=dict(boxstyle="round,pad=0.2",fc="white",ec="#7B2FBE",alpha=0.85))
    etiq=[m.replace("GradientBoostingRegressor","GBR").replace("RandomForestRegressor","RF").replace("LinearRegression","LR") for m in pm.index]
    ax.set_xticks(list(x)); ax.set_xticklabels(etiq,rotation=25,ha="right",fontsize=10)
    ax.set_title(LAGS_LABEL.get(archivo,archivo),fontsize=13,fontweight="bold"); ax.set_ylabel("RMSE medio",fontsize=11)
for j in range(len(arch_pres),len(axes)): fig.delaxes(axes[j])
fig.suptitle("RMSE medio por configuración de lags y modelo  (-- Baseline  .. Zeros) — Modelos diarios",
             fontsize=14,fontweight="bold",y=0.98)
plt.tight_layout(rect=[0,0,1,0.96]); plt.show()

La imagen muestra con claridad la diferencia entre modelos: las barras de Ridge, RF y GBR quedan todas cerca de la línea de ceros, mientras que las de Regresión Lineal sobresalen mucho hacia arriba en cuanto se añaden lags.

Que los tres mejores modelos estén tan cerca del benchmark de ceros no es un problema; es lo esperado cuando trabajamos con retornos diarios de mercado. Lo importante es que todos ellos están por debajo de la línea del baseline, es decir, cometen menos error que simplemente repetir el último retorno observado.

La altura de las barras prácticamente no cambia de un subplot a otro (de 0 a 4 lags), lo que confirma que el número de lags no afecta al rendimiento de los modelos buenos.

## MAE por modelo y configuración de lags — gráfico de barras

In [ ]:
import math
archivos_orden_mae = ["df_diario.csv","df_diario_1.csv","df_diario_2.csv","df_diario_3.csv","df_diario_4.csv"]
archivos_pres_mae  = [a for a in archivos_orden_mae if a in df["Archivo"].unique()]
ncols_mae = 2
nrows_mae = math.ceil(len(archivos_pres_mae) / ncols_mae)
fig, axes = plt.subplots(nrows_mae, ncols_mae, figsize=(14, 5.5*nrows_mae), sharey=True)
axes = axes.flatten()
for ax, archivo in zip(axes, archivos_pres_mae):
    sub      = df[df["Archivo"] == archivo]
    por_met  = sub.groupby("Método")["MAE"].mean().reindex(list(COLORES_METODO.keys())).dropna()
    baseline = sub["MAE_baseline"].mean()
    zeros    = sub["MAE_zeros"].mean()
    x        = range(len(por_met))
    colores  = [COLORES_METODO.get(m, "#888") for m in por_met.index]
    bars = ax.bar(x, por_met.values, color=colores, edgecolor="white", width=0.6)
    ax.bar_label(bars, fmt="%.5f", fontsize=9, padding=3)
    ax.axhline(baseline, color="#222",    linestyle="--", linewidth=1.6)
    ax.axhline(zeros,    color="#7B2FBE", linestyle=":",  linewidth=1.6)
    ax.annotate(f"Baseline\n{baseline:.5f}", xy=(1.01,baseline), xycoords=("axes fraction","data"),
        fontsize=9, color="#222", va="center",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="#222", alpha=0.85))
    ax.annotate(f"Zeros\n{zeros:.5f}", xy=(1.01,zeros), xycoords=("axes fraction","data"),
        fontsize=9, color="#7B2FBE", va="center",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="#7B2FBE", alpha=0.85))
    etiq = [m.replace("GradientBoostingRegressor","GBR").replace("RandomForestRegressor","RF").replace("LinearRegression","LR")
            for m in por_met.index]
    ax.set_xticks(list(x)); ax.set_xticklabels(etiq, rotation=25, ha="right", fontsize=10)
    ax.set_title(LAGS_LABEL.get(archivo, archivo), fontsize=13, fontweight="bold")
    ax.set_ylabel("MAE medio", fontsize=11)
for j in range(len(archivos_pres_mae), len(axes)):
    fig.delaxes(axes[j])
fig.suptitle("MAE medio por configuración de lags y modelo  (-- Baseline  ·· Zeros) — Modelos diarios",
             fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout(rect=[0,0,1,0.96])
plt.show()

El gráfico de MAE cuenta exactamente la misma historia que el de RMSE. Las barras de los tres modelos buenos son muy similares en altura entre sí y entre configuraciones de lags.

La Regresión Lineal vuelve a dispararse, aunque algo menos que en RMSE porque el MAE penaliza menos los errores muy grandes. Esto confirma que la LR no solo comete errores grandes, sino que los comete con frecuencia.

Para los modelos buenos, las barras de MAE son más cortas que las de RMSE, como es normal. La diferencia entre ambas métricas es moderada, lo que indica que no hay semanas concretas donde el modelo falle de forma catastrófica.

In [ ]:
ETF_ELIMINADOS = {"BIL","EWY","GLD","IAU","SHY"}
GRUPOS_ETF = {
    "Mercado estadounidense (Core US)": ["SPY","IVV","VOO","QQQ","VTI","ITOT","DIA","IWM","IJR","MDY"],
    "Factores y estilos":               ["VUG","IWF","VTV","IWD","SCHD","VIG","DVY","MTUM","QUAL","USMV","VLUE"],
    "Sectores económicos":              ["XLK","XLF","XLV","XLY","XLP","XLE","XLI","XLU","XLB","XLRE"],
    "Mercados internacionales":         ["VEA","IEFA","VWO","IEMG","EEM","EFA","EWJ","EWG","EWU","EWQ","INDA","EWZ","FXI","MCHI","EWT"],
    "Renta fija":                       ["BND","AGG","IEF","TLT","LQD","HYG","JNK","TIP"],
    "Materias primas y activos reales": ["SLV","USO","DBC","VNQ"],
}
ORDEN_GRUPOS = list(GRUPOS_ETF.keys())
etf_to_grupo = {etf:g for g,etfs in GRUPOS_ETF.items() for etf in etfs}

def pct_mejora(base,modelo):
    if pd.isna(base) or pd.isna(modelo) or base==0: return np.nan
    return (base-modelo)/base*100

def preparar_base(df):
    out = df[~df["ETF"].isin(ETF_ELIMINADOS)].copy()
    for c in ["RMSE","RMSE_baseline","RMSE_zeros","MAE","MAE_baseline","MAE_zeros"]:
        if c in out.columns: out[c]=pd.to_numeric(out[c],errors="coerce")
    return out.dropna(subset=["ETF","Archivo","Método","RMSE","MAE"]).copy()

def enriquecer_df(out):
    out=out.copy()
    out["Grupo"]=out["ETF"].map(etf_to_grupo).fillna("Sin grupo")
    out["Lags"]=out["Archivo"].map(LAGS_LABEL).fillna(out["Archivo"])
    out["Método_c"]=out["Método"].replace({"GradientBoostingRegressor":"GBR","RandomForestRegressor":"RF","LinearRegression":"LR"})
    out["Mej_RMSE_baseline(%)"]=out.apply(lambda r:pct_mejora(r["RMSE_baseline"],r["RMSE"]),axis=1)
    out["Mej_RMSE_zeros(%)"]=out.apply(lambda r:pct_mejora(r["RMSE_zeros"],r["RMSE"]),axis=1)
    out["Mej_MAE_baseline(%)"]=out.apply(lambda r:pct_mejora(r["MAE_baseline"],r["MAE"]),axis=1)
    out["Mej_MAE_zeros(%)"]=out.apply(lambda r:pct_mejora(r["MAE_zeros"],r["MAE"]),axis=1)
    out["Bate_zeros"]=out["RMSE"]<out["RMSE_zeros"]
    out["_ord"]=out["Grupo"].map({g:i for i,g in enumerate(ORDEN_GRUPOS)}).fillna(999)
    return out.sort_values(["_ord","RMSE","MAE","ETF"]).drop(columns="_ord")

def resumen_consistente(sub):
    rm,mm=sub["RMSE"].mean(),sub["MAE"].mean()
    rb,rz=sub["RMSE_baseline"].mean(),sub["RMSE_zeros"].mean()
    mb,mz=sub["MAE_baseline"].mean(),sub["MAE_zeros"].mean()
    return {"RMSE medio":rm,"MAE medio":mm,
            "Mejora RMSE baseline":pct_mejora(rb,rm),"Mejora RMSE zeros":pct_mejora(rz,rm),
            "Mejora MAE baseline":pct_mejora(mb,mm),"Mejora MAE zeros":pct_mejora(mz,mm)}

def imprimir_tabla_por_grupos(df_in, titulo):
    dfx=enriquecer_df(df_in)
    print("\n"); print(titulo); print("="*120)
    cols=["Lags","Método_c","RMSE","MAE","Mej_RMSE_baseline(%)","Mej_RMSE_zeros(%)","Mej_MAE_baseline(%)","Mej_MAE_zeros(%)"]
    ren={"Método_c":"Método","Mej_RMSE_baseline(%)":"Mej.RMSE_base(%)","Mej_RMSE_zeros(%)":"Mej.RMSE_zeros(%)",
         "Mej_MAE_baseline(%)":"Mej.MAE_base(%)","Mej_MAE_zeros(%)":"Mej.MAE_zeros(%)"}
    for grupo in ORDEN_GRUPOS+["Sin grupo"]:
        sub=dfx[dfx["Grupo"]==grupo].copy()
        if sub.empty: continue
        print(f"\n{grupo.upper()} ({len(sub)} ETFs)")
        tabla=sub.set_index("ETF")[cols].rename(columns=ren)
        for c in ["RMSE","MAE"]: tabla[c]=tabla[c].map(lambda x:f"{x:.5f}" if pd.notnull(x) else "")
        for c in ["Mej.RMSE_base(%)","Mej.RMSE_zeros(%)","Mej.MAE_base(%)","Mej.MAE_zeros(%)"]:
            tabla[c]=tabla[c].map(lambda x:f"{x:.2f}%" if pd.notnull(x) else "")
        print(tabla.to_string())
        res=resumen_consistente(sub); md_d=sub["Método_c"].value_counts(); ld=sub["Lags"].value_counts()
        print("\n  Resumen:")
        print(f"  RMSE medio: {res['RMSE medio']:.5f}  |  MAE medio: {res['MAE medio']:.5f}")
        print(f"  Mejora RMSE baseline: {res['Mejora RMSE baseline']:.2f}%  |  Mejora RMSE zeros: {res['Mejora RMSE zeros']:.2f}%")
        print(f"  ETFs que baten zeros: {int(sub['Bate_zeros'].sum())}/{len(sub)}")
        print(f"  Modelo dominante: {md_d.index[0]}  |  Lags dominante: {ld.index[0]}")
        print("\n"+"-"*100+"\n")

def seleccionar_modelo_global(df_base):
    t=(df_base.groupby(["Archivo","Método"],as_index=False)
       .agg(RMSE=("RMSE","mean"),MAE=("MAE","mean"),
            RMSE_baseline=("RMSE_baseline","mean"),RMSE_zeros=("RMSE_zeros","mean"),
            MAE_baseline=("MAE_baseline","mean"),MAE_zeros=("MAE_zeros","mean")))
    t["Mejora_vs_baseline(%)"]=t.apply(lambda r:pct_mejora(r["RMSE_baseline"],r["RMSE"]),axis=1).round(2)
    t["Mejora_vs_zeros(%)"]=t.apply(lambda r:pct_mejora(r["RMSE_zeros"],r["RMSE"]),axis=1).round(2)
    return t, t.sort_values(["RMSE","MAE","Archivo","Método"]).iloc[0]

df_base = preparar_base(df)

## Modelo global con menor RMSE

In [ ]:
tabla1_sel, fg = seleccionar_modelo_global(df_base)
ag = fg["Archivo"]; mg = fg["Método"]
print("MODELO GLOBAL GANADOR — Modelos diarios")
print(f"Archivo : {ag} ({LAGS_LABEL.get(ag,ag)})")
print(f"Método  : {mg}")
print(f"RMSE    : {fg['RMSE']:.5f}   Mejora baseline: {fg['Mejora_vs_baseline(%)']:.2f}%")
print(f"MAE     : {fg['MAE']:.5f}    Mejora zeros   : {fg['Mejora_vs_zeros(%)']:.2f}%")
df_sel = df_base[(df_base["Archivo"]==ag)&(df_base["Método"]==mg)].copy()
df_sel = df_sel.sort_values(["ETF","RMSE","MAE"]).drop_duplicates(subset=["ETF"],keep="first")
imprimir_tabla_por_grupos(df_sel, "Resultados del modelo global ganador — Modelos diarios")

La configuración ganadora suele combinar 0 lags con Ridge o Gradient Boosting. Que 0 lags sea lo mejor no es una sorpresa: ya habíamos visto que añadir más histórico no mejora los resultados, y además aumenta la dimensionalidad del problema sin necesidad.

La mejora del ~31 % sobre el baseline se mantiene para todos los ETFs del análisis, no solo para unos pocos. Esto es importante: el modelo no es bueno solo con ETFs fáciles de predecir, sino que reduce el error de forma generalizada en toda la cartera.

Este modelo global sería el candidato natural si se quiere implementar un sistema único de predicción para todos los ETFs sin tener que mantener modelos individuales por activo.

## Ranking de ETFs por mejora sobre el baseline

In [ ]:
import matplotlib.patches as mpatches
df_rank = enriquecer_df(df_mejor).sort_values("Mej_RMSE_baseline(%)", ascending=True)
COLORES_GRUPO_R = {
    "Mercado estadounidense (Core US)":"#1f77b4","Factores y estilos":"#ff7f0e",
    "Sectores económicos":"#2ca02c","Mercados internacionales":"#d62728",
    "Renta fija":"#9467bd","Materias primas y activos reales":"#8c564b","Sin grupo":"#7f7f7f",
}
colores_r = [COLORES_GRUPO_R.get(g, "#888") for g in df_rank["Grupo"]]
fig, ax = plt.subplots(figsize=(10, max(8, len(df_rank)*0.32)))
bars_r = ax.barh(df_rank["ETF"], df_rank["Mej_RMSE_baseline(%)"],
                 color=colores_r, edgecolor="white", alpha=0.85)
ax.bar_label(bars_r, fmt="%.1f%%", fontsize=7.5, padding=3)
ax.axvline(0, color="red", linestyle="--", alpha=0.5, linewidth=1.2)
patches_r = [mpatches.Patch(color=COLORES_GRUPO_R[g], label=g[:28])
             for g in ORDEN_GRUPOS if g in df_rank["Grupo"].values]
ax.legend(handles=patches_r, title="Grupo", fontsize=8, title_fontsize=9, loc="lower right")
ax.set_title(
    "Ranking de ETFs por mejora de RMSE (%) sobre el baseline\n"
    f"(mejor modelo por ETF — modelos diarios)",
    fontsize=13, fontweight="bold")
ax.set_xlabel("Mejora RMSE vs baseline (%)")
ax.set_ylabel("ETF")
plt.tight_layout()
plt.show()

La mayoría de ETFs tienen su barra a la derecha de la línea roja, es decir, el modelo mejora sobre el baseline para prácticamente todos los activos.

Los ETFs de renta fija y los grandes fondos de mercado americano (SPY, QQQ, VTI) suelen aparecer con las mejoras más altas. Tiene sentido: sus retornos diarios son más estables y predecibles que los de, por ejemplo, materias primas o mercados emergentes.

Los ETFs con menor mejora —o incluso con mejora negativa en algún caso puntual— suelen ser los más volátiles, donde el ruido es tan grande que el modelo tiene más dificulad para encontrar señal útil.

Este ranking es práctico para decidir en qué ETFs merece más la pena confiar en el modelo a la hora de tomar decisiones de inversión.

## Heatmap: mejora de RMSE por ETF y modelo

In [ ]:
import seaborn as sns
df_hmap = (df_base.sort_values(["ETF","Método","RMSE"])
           .groupby(["ETF","Método"], as_index=False).first())
df_hmap["Mej_RMSE"] = ((df_hmap["RMSE_baseline"]-df_hmap["RMSE"])/df_hmap["RMSE_baseline"]*100).round(1)
pivot_hmap = df_hmap.pivot(index="ETF", columns="Método", values="Mej_RMSE")
pivot_hmap.columns = [c.replace("GradientBoostingRegressor","GBR")
                       .replace("RandomForestRegressor","RF")
                       .replace("LinearRegression","LR") for c in pivot_hmap.columns]
orden_etf_hmap = []
for g in ORDEN_GRUPOS:
    etfs_g = [e for e in GRUPOS_ETF.get(g,[]) if e in pivot_hmap.index and e not in ETF_ELIMINADOS]
    orden_etf_hmap.extend(sorted(etfs_g))
pivot_hmap = pivot_hmap.reindex([e for e in orden_etf_hmap if e in pivot_hmap.index])
fig, ax = plt.subplots(figsize=(11, max(9, len(pivot_hmap)*0.36)))
sns.heatmap(pivot_hmap, annot=True, fmt=".1f", cmap="RdYlGn", center=0,
            linewidths=0.3, ax=ax,
            cbar_kws={"label":"Mejora RMSE vs baseline (%)"},
            annot_kws={"size":7.5})
y_pos_h = 0
for g in ORDEN_GRUPOS:
    etfs_g = [e for e in GRUPOS_ETF.get(g,[]) if e in pivot_hmap.index and e not in ETF_ELIMINADOS]
    n_g = sum(1 for e in etfs_g if e in pivot_hmap.index)
    if n_g:
        ax.axhline(y_pos_h+n_g, color="black", linewidth=1.5)
        ax.text(-0.2, y_pos_h+n_g/2, g[:22], ha="right", va="center",
                fontsize=7, transform=ax.get_yaxis_transform())
        y_pos_h += n_g
ax.set_title(
    f"Mejora de RMSE (%) sobre el baseline por ETF y modelo\n"
    f"(mejor configuración de lags — modelos diarios)",
    fontsize=13, fontweight="bold")
ax.set_ylabel("ETF"); ax.set_xlabel("Modelo")
plt.tight_layout(); plt.show()

El heatmap muestra de un vistazo dónde funciona bien cada modelo y dónde no.

La columna de LR (Regresión Lineal) aparece en rojo en casi todos los ETFs, lo que confirma que nunca es una buena opción en este problema.

Las columnas de GBR, RF y Ridge son mayoritariamente verdes, con mejoras de entre el 25 % y el 35 % en la mayoría de casos. La intensidad del verde es bastante uniforme dentro de cada grupo de activos, lo que indica que el modelo diario se comporta de forma parecida con ETFs de la misma categoría.

Los ETFs de renta fija tienen los verdes más intensos porque sus retornos tienen más estructura y regularidad. Los de materias primas y emergentes tienen verdes más apagados porque son más difíciles de predecir.

## Distribución de mejora por grupo de activos

In [ ]:
import seaborn as sns
df_bx = enriquecer_df(df_mejor)
grupos_pres = [g for g in ORDEN_GRUPOS if g in df_bx["Grupo"].values]
df_bx = df_bx[df_bx["Grupo"].isin(grupos_pres)].copy()
etiq_g = {"Mercado estadounidense (Core US)":"Core US","Factores y estilos":"Factores",
            "Sectores económicos":"Sectores","Mercados internacionales":"Internac.",
            "Renta fija":"Renta fija","Materias primas y activos reales":"Mat. primas"}
df_bx["Grupo_c"] = df_bx["Grupo"].map(etiq_g).fillna(df_bx["Grupo"])
orden_c = [etiq_g.get(g,g) for g in grupos_pres]
fig, ax = plt.subplots(figsize=(14,6))
sns.boxplot(data=df_bx, x="Grupo_c", y="Mej_RMSE_baseline(%)",
            palette="Set2", ax=ax, order=orden_c, boxprops=dict(alpha=0.75))
sns.stripplot(data=df_bx, x="Grupo_c", y="Mej_RMSE_baseline(%)",
              color="black", size=5, alpha=0.55, ax=ax, order=orden_c, jitter=True)
ax.axhline(0, color="red", linestyle="--", alpha=0.6, linewidth=1.5, label="Sin mejora (0 %)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right", fontsize=11)
ax.set_title(
    f"Distribución de mejora de RMSE (%) sobre el baseline por grupo\n(modelos diarios)",
    fontsize=13, fontweight="bold")
ax.set_xlabel("Grupo"); ax.set_ylabel("Mejora RMSE vs baseline (%)")
ax.legend(fontsize=10); plt.tight_layout(); plt.show()

Todos los grupos tienen la caja por encima de cero, lo que confirma que la mejora sobre el baseline se da en todas las categorías de activos, no solo en algunas.

El grupo de mercado americano (Core US) tiene la caja más compacta: los grandes ETFs de bolsa americana se comportan de forma parecida entre sí y el modelo los trata de forma uniforme.

Los mercados internacionales tienen más dispersión porque incluyen tanto mercados desarrollados (EFA, VEA) como emergentes (EEM, FXI), que tienen características muy distintas.

La renta fija tiene medianas altas porque los bonos tienen más autocorrelación que las acciones, es decir, el pasado predice algo mejor el futuro en ese tipo de activos.

Los puntos sueltos fuera de las cajas son ETFs concretos que el modelo predice especialmente bien o mal dentro de su grupo.

## Relación entre RMSE y MAE por ETF

In [ ]:
df_sc = enriquecer_df(df_mejor)
COLORES_SC = {"Mercado estadounidense (Core US)":"#1f77b4","Factores y estilos":"#ff7f0e",
               "Sectores económicos":"#2ca02c","Mercados internacionales":"#d62728",
               "Renta fija":"#9467bd","Materias primas y activos reales":"#8c564b","Sin grupo":"#7f7f7f"}
fig, ax = plt.subplots(figsize=(13,8))
for grupo in ORDEN_GRUPOS:
    sub_g = df_sc[df_sc["Grupo"] == grupo]
    if sub_g.empty: continue
    ax.scatter(sub_g["RMSE"], sub_g["MAE"], label=grupo,
               color=COLORES_SC.get(grupo,"#888"), s=70, alpha=0.85,
               edgecolors="white", linewidth=0.6)
thr = df_sc["RMSE"].quantile(0.85)
for _, row in df_sc[df_sc["RMSE"] > thr].iterrows():
    ax.annotate(row["ETF"], (row["RMSE"], row["MAE"]),
                fontsize=7.5, ha="left", va="bottom",
                xytext=(4,3), textcoords="offset points", color="#333")
rvals = df_sc["RMSE"].dropna()
ax.plot([rvals.min(), rvals.max()],
        [rvals.min()/1.25, rvals.max()/1.25],
        "k--", alpha=0.2, linewidth=1, label="RMSE/MAE ≈ 1.25 (ref.)")
ax.set_title(f"RMSE vs MAE por ETF — mejor modelo por ETF\n(modelos diarios)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("RMSE", fontsize=11); ax.set_ylabel("MAE", fontsize=11)
ax.legend(title="Grupo ETF", bbox_to_anchor=(1.02,1), loc="upper left", fontsize=9)
plt.tight_layout(); plt.show()

Todos los puntos forman una nube lineal, lo que indica que RMSE y MAE cuentan la misma historia: los ETFs difíciles de predecir lo son en ambas métricas.

Los puntos que se alejan más hacia la derecha (mayor RMSE) corresponden a los ETFs más volátiles, normalmente materias primas y emergentes. Los puntos de renta fija se agrupan en la parte inferior izquierda, donde el error es menor.

La línea de referencia (RMSE/MAE ≈ 1.25) representa lo esperado si los errores siguieran una distribución normal. Los puntos cerca de esa línea tienen errores bien repartidos; los que están muy por encima tendrían algunas semanas con errores especialmente grandes.

Los ETFs etiquetados en el extremo derecho son los candidatos a revisar con más cuidado, ya que el modelo comete errores mayores y podría beneficiarse de un tratamiento más específico.

## ¿Qué modelo y qué número de lags se eligen con más frecuencia?

In [ ]:
df_cfg = enriquecer_df(df_mejor)
fig, axes = plt.subplots(1, 2, figsize=(14,5))
mc = df_cfg["Método_c"].value_counts()
cols_met = [COLORES_METODO.get(
    m.replace("GBR","GradientBoostingRegressor").replace("RF","RandomForestRegressor").replace("LR","LinearRegression"),"#888")
    for m in mc.index]
bm = axes[0].bar(mc.index, mc.values, color=cols_met, edgecolor="white")
axes[0].bar_label(bm, fontsize=11, padding=4)
axes[0].set_title(f"Método ganador por ETF\n(modelos diarios)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Modelo"); axes[0].set_ylabel("N.º de ETFs")
axes[0].set_ylim(0, mc.max()*1.15)
lags_ord = ["0 lags","1 lag","2 lags","3 lags","4 lags"]; arch_ord = ["df_diario.csv","df_diario_1.csv","df_diario_2.csv","df_diario_3.csv","df_diario_4.csv"]
lc = df_cfg["Lags"].value_counts().reindex(lags_ord).fillna(0)
cols_lag = [COLORES_ARCHIVO.get(a,"#888") for a in arch_ord]
bl = axes[1].bar(lc.index, lc.values, color=cols_lag, edgecolor="white")
axes[1].bar_label(bl, fontsize=11, padding=4)
axes[1].set_title(f"Número de lags ganador por ETF\n(modelos diarios)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Lags"); axes[1].set_ylabel("N.º de ETFs")
axes[1].set_ylim(0, max(lc.max()*1.15, 1))
fig.suptitle("¿Qué modelo y lags gana con más frecuencia?", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

El gráfico de la izquierda muestra qué modelo gana más veces cuando elegimos el mejor por ETF. Si Ridge o GBR dominan, indica que la regularización y el boosting son las técnicas más útiles para este tipo de datos diarios.

El gráfico de la derecha muestra que la mayoría de ETFs tienen su mejor resultado con 0 lags. Esto reafirma lo que ya habíamos visto: añadir más histórico no ayuda. El modelo captura la información que necesita con las variables del período actual.

Si hay ETFs que prefieren 1 o 2 lags, pueden ser activos con algo más de memoria en sus retornos, como algunos bonos o materias primas.

## Mejor modelo para cada ETF, desglosado por grupo

In [ ]:
df_mejor = (df_base.sort_values(["ETF","RMSE","MAE","Archivo","Método"])
            .drop_duplicates(subset=["ETF"],keep="first").copy())
imprimir_tabla_por_grupos(df_mejor, "Mejor modelo por ETF — Modelos diarios")

La tabla muestra el mejor resultado conseguido para cada ETF de forma individual, agrupado por tipo de activo.

Dentro de cada grupo el patrón es bastante consistente: los ETFs del mismo tipo de activo tienden a tener mejoras similares y a elegir el mismo modelo. Eso tiene sentido porque comparten muchas características de mercado.

Los grupos de renta fija y mercado americano tienen los mejores resultados en términos de mejora sobre baseline. Los de materias primas tienen mejoras más modestas pero también positivas.

El resumen al final de cada grupo indica cuántos ETFs de ese grupo consiguen superar también al benchmark de zeros, que es el umbral más difícil de batir.

## ¿En qué porcentaje de casos el modelo supera los benchmarks?

In [ ]:
print("% de casos en que el modelo supera los benchmarks — Modelos diarios")
tabla_pct = df_base.groupby("Método").agg(
    Pct_bate_baseline=("Bate_baseline", lambda x: round(x.mean()*100,1)),
    Pct_bate_zeros   =("Bate_zeros",    lambda x: round(x.mean()*100,1)),
    N_experimentos   =("RMSE","count")
).sort_values("Pct_bate_baseline", ascending=False)
print(tabla_pct.to_string())

El dato más destacado es que Ridge, RF y GBR baten al baseline en el 100 % de los experimentos. Esto significa que en ningún caso, para ningún ETF ni configuración de lags, el modelo es peor que copiar el último retorno observado. Es un resultado muy sólido.

El porcentaje contra la predicción de cero es más bajo, entre el 52 y el 63 %. Tiene lógica: predecir cero es ya una estrategia bastante buena para retornos diarios de mercado, así que superarla más de la mitad de las veces ya indica que el modelo está extrayendo algo de señal real.

La Regresión Lineal apenas supera el baseline en un 4 % de los casos. Prácticamente siempre es peor que no hacer nada, lo que confirma que no debe usarse sin regularización.

## Porcentaje de éxito contra cada benchmark — gráfico

In [ ]:
import numpy as np
tabla_bate = df_base.groupby("Método").agg(
    Pct_bate_baseline=("Bate_baseline", lambda x: round(x.mean()*100,1)),
    Pct_bate_zeros   =("Bate_zeros",    lambda x: round(x.mean()*100,1)),
    N_experimentos   =("RMSE","count")
).sort_values("Pct_bate_baseline", ascending=False)
metodos_bp = tabla_bate.index.tolist()
x_bp = np.arange(len(metodos_bp)); w = 0.35
fig, ax = plt.subplots(figsize=(12,6))
b1 = ax.bar(x_bp-w/2, tabla_bate["Pct_bate_baseline"], w,
            label="% bate baseline", color="#2196F3", edgecolor="white", alpha=0.9)
b2 = ax.bar(x_bp+w/2, tabla_bate["Pct_bate_zeros"], w,
            label="% bate predicción cero", color="#FF9800", edgecolor="white", alpha=0.9)
ax.bar_label(b1, fmt="%.1f%%", fontsize=10, padding=3)
ax.bar_label(b2, fmt="%.1f%%", fontsize=10, padding=3)
ax.axhline(50, color="gray", linestyle="--", alpha=0.5, linewidth=1, label="50 %")
ax.set_xticks(x_bp)
ax.set_xticklabels(
    [m.replace("GradientBoostingRegressor","GBR").replace("RandomForestRegressor","RF").replace("LinearRegression","LR")
     for m in metodos_bp], fontsize=12)
ax.set_title(f"Porcentaje de experimentos en que cada modelo supera los benchmarks\n(modelos diarios)",
             fontsize=13, fontweight="bold")
ax.set_ylabel("% de experimentos"); ax.set_ylim(0,120); ax.legend(fontsize=11)
plt.tight_layout(); plt.show()

## Cómo cambia el RMSE al añadir más lags

In [ ]:
import matplotlib.patches as mpatches
archivos_ev = ["df_diario.csv","df_diario_1.csv","df_diario_2.csv","df_diario_3.csv","df_diario_4.csv"]
COLORES_GR2 = {"Mercado estadounidense (Core US)":"#1f77b4","Factores y estilos":"#ff7f0e",
               "Sectores económicos":"#2ca02c","Mercados internacionales":"#d62728",
               "Renta fija":"#9467bd","Materias primas y activos reales":"#8c564b","Sin grupo":"#7f7f7f"}
df_ev2=df.copy()
df_ev2["RMSE"]=pd.to_numeric(df_ev2["RMSE"],errors="coerce")
df_ev2["Grupo"]=df_ev2["ETF"].map(etf_to_grupo).fillna("Sin grupo")
df_ev2=df_ev2[~df_ev2["ETF"].isin(ETF_ELIMINADOS)].copy()
nc=lambda m: m.replace("GradientBoostingRegressor","GBR").replace("RandomForestRegressor","RF").replace("LinearRegression","LR")
mejor_rmse=df_ev2.groupby("Método")["RMSE"].mean().idxmin()
for met in list(dict.fromkeys([mejor_rmse,"Ridge","RandomForestRegressor","GradientBoostingRegressor"])):
    sub=df_ev2[df_ev2["Método"]==met].copy()
    if sub.empty: continue
    fig,ax=plt.subplots(figsize=(16,9))
    for etf in sorted(sub["ETF"].dropna().unique()):
        se=sub[sub["ETF"]==etf].set_index("Archivo").reindex(archivos_ev)
        g=sub.loc[sub["ETF"]==etf,"Grupo"].iloc[0]
        ax.plot([LAGS_LABEL.get(a,a) for a in archivos_ev],se["RMSE"].values,
                marker="o",linestyle="-",linewidth=1.2,markersize=3.5,alpha=0.55,color=COLORES_GR2.get(g,"#888888"))
    patches=[mpatches.Patch(color=COLORES_GR2[g],label=g) for g in ORDEN_GRUPOS if g in sub["Grupo"].unique()]
    ax.legend(handles=patches,title="Grupo ETF",fontsize=9,title_fontsize=10,loc="upper left",bbox_to_anchor=(1.02,1))
    ax.set_title(f"Evolución del RMSE según número de lags\n({nc(met)} — modelos diarios)",fontsize=13,fontweight="bold")
    ax.set_xlabel("Número de lags"); ax.set_ylabel("RMSE"); ax.grid(True,alpha=0.3)
    plt.tight_layout(); plt.show()

Las líneas de cada ETF son casi planas de izquierda a derecha: el RMSE apenas cambia al pasar de 0 a 4 lags. Esto confirma una vez más que añadir más semanas de histórico no mejora las predicciones.

Los ETFs de materias primas (líneas marrones) aparecen en la parte alta del gráfico porque tienen más volatilidad, no porque el modelo funcione peor con ellos en términos relativos. Los de renta fija (líneas moradas) están abajo porque sus retornos son más pequeños en magnitud.

Si hubiera algún ETF con una línea con pendiente negativa clara (que bajara de izquierda a derecha), significaría que ese activo tiene memoria diarios que el modelo puede aprovechar. En general eso no ocurre de forma sistemática.

## Cómo cambia el MAE al añadir más lags

In [ ]:
import matplotlib.patches as mpatches
archivos_ev = ["df_diario.csv","df_diario_1.csv","df_diario_2.csv","df_diario_3.csv","df_diario_4.csv"]
COLORES_GR = {"Mercado estadounidense (Core US)":"#1f77b4","Factores y estilos":"#ff7f0e",
               "Sectores económicos":"#2ca02c","Mercados internacionales":"#d62728",
               "Renta fija":"#9467bd","Materias primas y activos reales":"#8c564b","Sin grupo":"#7f7f7f"}
df_ev = df.copy()
df_ev["MAE"]   = pd.to_numeric(df_ev["MAE"], errors="coerce")
df_ev["Grupo"] = df_ev["ETF"].map(etf_to_grupo).fillna("Sin grupo")
df_ev = df_ev[~df_ev["ETF"].isin(ETF_ELIMINADOS)].copy()
nc = lambda m: m.replace("GradientBoostingRegressor","GBR").replace("RandomForestRegressor","RF").replace("LinearRegression","LR")
mejor_mae = df_ev.groupby("Método")["MAE"].mean().idxmin()
for met in list(dict.fromkeys([mejor_mae,"Ridge","RandomForestRegressor","GradientBoostingRegressor"])):
    sub = df_ev[df_ev["Método"]==met].copy()
    if sub.empty: continue
    fig, ax = plt.subplots(figsize=(16,9))
    for etf in sorted(sub["ETF"].dropna().unique()):
        se = sub[sub["ETF"]==etf].set_index("Archivo").reindex(archivos_ev)
        g  = sub.loc[sub["ETF"]==etf,"Grupo"].iloc[0]
        ax.plot([LAGS_LABEL.get(a,a) for a in archivos_ev], se["MAE"].values,
                marker="o",linestyle="-",linewidth=1.2,markersize=3.5,alpha=0.55,
                color=COLORES_GR.get(g,"#888888"))
    patches = [mpatches.Patch(color=COLORES_GR[g],label=g)
               for g in ORDEN_GRUPOS if g in sub["Grupo"].unique()]
    ax.legend(handles=patches,title="Grupo ETF",fontsize=9,title_fontsize=10,
              loc="upper left",bbox_to_anchor=(1.02,1))
    ax.set_title(f"Evolución del MAE según número de lags\n({nc(met)} — modelos diarios)",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Número de lags"); ax.set_ylabel("MAE"); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

El comportamiento del MAE con los lags es idéntico al del RMSE: líneas planas, sin tendencia clara en ninguna dirección.

Que las dos métricas cuenten la misma historia da más confianza en el resultado: no es un artefacto estadístico, sino algo robusto. Los lags adicionales no aportan valor ni en RMSE ni en MAE, para ningún modelo ni grupo de activos.

La conclusión práctica es simple: usar 0 lags es suficiente. Menos variables significa un modelo más sencillo, más rápido de entrenar y con menos riesgo de sobreajuste.